# Implicit Decision Gate: a verified long-running agent walkthrough

## Premise

**Actors:** human job owner and system designer.

Implicit Decision Gate is a deliberately small, fictional contract-completion stage inside the trust architecture described in 1Password's [Verified Loops](https://1password.com/blog/verified-loops-building-ai-agent-trust). That architecture makes the human-owned job definition the verification boundary and leaves humans the consequential judgments that cannot be verified mechanically. This notebook makes one such boundary executable: what happens when a system-observed effect reveals that the request never made a required choice?

Imagine a service behind 1Password item-sharing links. A customer can share a 1Password item, such as a Login item containing a password, by link. A brief asks a coding agent to make newly created links expire after 30 days. The fictional service stores link records in a PostgreSQL table named `public.share_links`.

The brief does not say what should happen to links customers already created. A valid PostgreSQL migration must nevertheless choose between two materially different outcomes:

| Decision | Existing links | New links |
| --- | --- | --- |
| `PRESERVE_EXISTING` | Keep their current non-expiring behavior | Expire after 30 days |
| `EXPIRE_EXISTING` | Expire 30 days after migration | Expire after 30 days |

Either policy could be legitimate. Expiring old links can break customer workflows; preserving them can retain access longer than the new policy intends. The agent should not silently invent the answer.

This scenario is fictional. It makes no claim about 1Password's production services, database schema, or implementation of item sharing. In `public.share_links`, `public` is only the standard PostgreSQL schema namespace. It does not mean the table, its rows, or the links are publicly accessible.

The fictional framing belongs to this explanatory notebook. The authoritative brief inspected below is written as an ordinary engineering request and is passed to Codex verbatim; it contains no disclaimer telling the agent that it is participating in a demo.


## Motivation

**Actor:** system designer.

This notebook follows one job across a durable agent loop: pin the human-owned inputs, run a fresh coding process, inspect its SQL, verify actual database behavior, ask a narrow evidence question against the brief, pause when intent is missing, record a typed owner decision, resume with a fresh coding process, and deterministically verify the result. Evidence comes from PostgreSQL, not from the coding agent's account of its own work.

The notebook drives the same public `idg` CLI as the terminal demo and then opens its persisted artifacts. It is an observability layer, not a second implementation of the orchestrator. `start` performs coding, probing, and evidence review as one operation; the later cells unpack those persisted stages in causal order.

Execute the state-changing `start`, `answer`, and `resume` cells once, from top to bottom. They are intentionally non-idempotent. To repeat the walkthrough, rerun from `start` and continue with the new run identifier.

This checked-in copy includes outputs from one genuine live execution. Rerunning it creates a new run with new identifiers, timestamps, SQL, and model classifications; the first migration may choose either supported rollout policy.

> The current implementation does not compile the brief into a general semantic contract. It asks one domain-specific question about one behavior observed at runtime. This notebook makes that boundary visible.


## System context

**Actor:** system designer.

The prototype completes one missing part of a human-owned contract and returns the result to the wider verified loop. Authenticated identities, controlled tools, attributable evidence, and permission enforcement remain responsibilities of the surrounding Verified Loops architecture.

```mermaid
flowchart LR
    owner[Human job owner]
    git[Git pinned brief and schema]
    orchestrator[Gate orchestrator]
    coder[Fresh Codex coding process]
    postgres[Disposable PostgreSQL verifier]
    reviewer[Fresh Codex evidence reviewer]
    gate[Deterministic logic inside orchestrator]
    store[Durable run store]
    verified[Return to wider verified loop]
    trust[Verified Loops trust controls]

    owner -->|Authoritative brief| git
    git -->|Pinned inputs| orchestrator
    orchestrator -->|Rendered coding prompt| coder
    coder -->|Structured migration SQL| orchestrator
    orchestrator -->|Schema and migration| postgres
    postgres -->|Normalized observations| orchestrator
    orchestrator -->|Brief and observed behavior| reviewer
    reviewer -->|Evidence classification| orchestrator
    orchestrator -->|Evidence and typed facts| gate
    gate -->|State transition| orchestrator
    orchestrator -->|State prompts evidence artifacts| store
    orchestrator -->|Missing judgment| owner
    owner -->|Typed contract amendment| orchestrator
    orchestrator -->|Verified contract completion| verified
    trust -.->|Identity tools evidence and permissions| orchestrator
```


## 1. Establish the notebook boundary

**Actor:** notebook operator.

This cell locates the repository and defines small display helpers. All state-changing work still goes through `uv run idg`; the helpers only execute commands and read the files that the application persists.


In [ ]:
from __future__ import annotations

import copy
import difflib
import hashlib
import json
import os
import shlex
import shutil
import stat
import subprocess
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "examples/share-link-expiration/brief.md"
        ).is_file():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from inside the implicit-decision-gate repository")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
COMMAND_ENV = os.environ.copy()
COMMAND_ENV.pop("VIRTUAL_ENV", None)


def run_command(
    arguments: list[str],
    *,
    show_output: bool = True,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        arguments,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
        env=COMMAND_ENV,
    )
    if show_output:
        print(f"$ {shlex.join(arguments)}")
        if completed.stdout:
            print(completed.stdout.rstrip())
        if completed.stderr:
            print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command exited with status {completed.returncode}: {shlex.join(arguments)}"
        )
    return completed


def run_idg(*arguments: str) -> dict[str, Any]:
    completed = run_command(["uv", "run", "idg", *arguments], check=False)
    try:
        payload = json.loads(completed.stdout)
    except json.JSONDecodeError as error:
        raise RuntimeError("idg did not return its expected JSON summary") from error
    if completed.returncode != 0:
        print(
            f"idg exited with status {completed.returncode}; "
            "its persisted summary remains inspectable"
        )
    return payload


def load_run(run_id: str) -> dict[str, Any]:
    path = REPO_ROOT / ".idg" / "runs" / run_id / "run.json"
    return json.loads(path.read_text(encoding="utf-8"))


print(f"Repository: {REPO_ROOT}")

## 2. Check the execution environment

**Actor:** notebook operator.

The live path needs Git and `uv`, an installed and authenticated Codex CLI, and PostgreSQL 17. Docker is required only when it supplies that disposable PostgreSQL instance; an external admin DSN can be provided with `IDG_POSTGRES_ADMIN_DSN`.

The application, rather than the operator's Codex configuration, pins every model process to `gpt-5.6-terra` with `xhigh` reasoning. The CLI version and the pinned settings are different facts: the version identifies the local harness, while the model and reasoning effort identify the requested inference configuration. Each invocation is recorded in `run.json`.


In [ ]:
from implicit_decision_gate.codex_client import CODEX_MODEL, CODEX_REASONING_EFFORT

EXTERNAL_POSTGRES = bool(os.environ.get("IDG_POSTGRES_ADMIN_DSN"))
required_tools = ["git", "uv", "codex"]
if not EXTERNAL_POSTGRES:
    required_tools.append("docker")

tool_paths = {name: shutil.which(name) for name in required_tools}
print(json.dumps(tool_paths, indent=2))
missing_tools = [name for name, path in tool_paths.items() if path is None]
if missing_tools:
    raise RuntimeError(f"Missing required tools: {', '.join(missing_tools)}")

run_command(["git", "rev-parse", "HEAD"])
run_command(["uv", "--version"])
run_command(["codex", "--version"])
print("Application-pinned model configuration:")
print(
    json.dumps(
        {"model": CODEX_MODEL, "reasoning_effort": CODEX_REASONING_EFFORT},
        indent=2,
    )
)
if EXTERNAL_POSTGRES:
    print("PostgreSQL source: IDG_POSTGRES_ADMIN_DSN is configured; its value is hidden.")
else:
    run_command(["docker", "compose", "version"])

## 3. Start the disposable verifier

**Actors:** notebook operator and PostgreSQL verifier.

When no external DSN is configured, Compose starts PostgreSQL 17. PostgreSQL is part of the trust argument: it executes the real DDL and exposes default, backfill, nullability, and rollback behavior that SQL text inspection alone cannot establish.


In [ ]:
if EXTERNAL_POSTGRES:
    print("Using the configured disposable PostgreSQL instance.")
else:
    run_command(["docker", "compose", "up", "-d", "--wait"])

## 4. Inspect the authoritative inputs

**Actors:** job owner for the brief, service repository for the baseline schema, and Git for provenance.

The brief and schema below are read from the current commit, not from uncommitted working-tree files. The seeded `existing-fixture` row is what makes the missing rollout policy observable. At this point the brief is exact source text; there is no precomputed semantic representation.


In [ ]:
HEAD_BEFORE_START = run_command(["git", "rev-parse", "HEAD"], show_output=False).stdout.strip()
BRIEF_PATH = "examples/share-link-expiration/brief.md"
SCHEMA_PATH = "examples/share-link-expiration/schema.sql"
brief_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{BRIEF_PATH}"], show_output=False
).stdout
schema_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{SCHEMA_PATH}"], show_output=False
).stdout

print(f"Pinned candidate commit: {HEAD_BEFORE_START}")
print("\nAuthoritative brief:\n")
print(brief_at_head)
print("Baseline schema and fixture:\n")
print(schema_at_head)

### Brief, context, and rendered prompt

**Actors:** human job owner, service repository, and gate prompt renderer.

The brief is the engineering ticket. The rendered prompt is not a second ticket and is not an engineer's manual reinterpretation of it. The gate application deterministically combines independently owned artifacts into the complete input for one ephemeral Codex process.

| Artifact | Owner | Role |
| --- | --- | --- |
| Authoritative brief | Human job owner | Product intent and verification boundary |
| Baseline schema | Service repository, pinned by Git | Technical context and observable fixture |
| Prompt envelope | Gate application | Execution isolation and structured-output instructions |
| Rendered prompt | Gate prompt renderer | Materialized envelope, verbatim brief, schema, and any approved amendment |
| Owner amendment | Human job owner | The smallest missing judgment added to attempt two |

The brief appears separately and inside each applicable prompt for two reasons. First, every Codex process is ephemeral and must receive its complete input. Second, `run.json` retains both the source contract and the exact materialized prompt so an auditor can verify that the application did not silently translate or replace the owner's words.

The prompt envelope contains execution and output constraints, not a second copy of the product requirements. Attempt two additionally contains the typed owner decision, its required behavior, and the PostgreSQL acceptance criteria derived from that explicit amendment.


## Lifecycle at a glance

**Actors:** human job owner, gate orchestrator, coding agents, PostgreSQL verifier, evidence reviewer, deterministic gate, and run store.

This swimlane shows the reference path that the remaining cells unpack. Attempt two is a new process, not a continuation of attempt one.

```mermaid
sequenceDiagram
    participant O as Human job owner
    participant G as Gate orchestrator
    participant C1 as Codex coder attempt one
    participant P as PostgreSQL verifier
    participant R as Codex evidence reviewer
    participant D as Deterministic gate
    participant S as Durable run store
    participant C2 as Codex coder attempt two

    O->>G: Supply authoritative brief
    G->>S: Persist pinned brief and commit
    G->>C1: Send rendered prompt
    C1-->>G: Return structured migration SQL
    G->>S: Persist prompt SQL and digest
    G->>P: Apply baseline and migration
    P-->>G: Return normalized rollout option
    G->>R: Send brief and observed behavior
    R-->>G: Return evidence classification
    G->>D: Evaluate evidence and typed facts
    D-->>G: Return AWAITING_OWNER
    G->>S: Persist AWAITING_OWNER
    G-->>O: Request one typed rollout choice
    O->>G: Select owner option
    G->>S: Persist READY_TO_RESUME
    G->>C2: Send fresh prompt with amendment
    C2-->>G: Return regenerated migration SQL
    G->>P: Apply baseline and regenerated migration
    P-->>G: Return normalized rollout option
    G->>D: Compare observed and selected options
    D-->>G: Return COMPLETED or FAILED
    G->>S: Persist COMPLETED or FAILED
```


## 5. Start one durable run

**Actor:** gate orchestrator.

`start` pins the current commit, creates a clean detached worktree, invokes a fresh Codex process for SQL, probes that SQL in PostgreSQL, and invokes a separate fresh Codex process for the evidence question. It returns only after those stages are persisted. The following cells inspect the captured start snapshot in their causal order. Because the model calls are live, an unexpected terminal state is displayed and stops this walkthrough instead of being forced into the expected story; rerun this cell to create another independent run.


In [ ]:
start_summary = run_idg("start")
RUN_ID = str(start_summary["run_id"])
RUN_DIR = REPO_ROOT / ".idg" / "runs" / RUN_ID
START_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))

if START_SNAPSHOT["state"] != "AWAITING_OWNER":
    raise RuntimeError(
        "This live run did not enter AWAITING_OWNER. Inspect the summary above; "
        "model output is nondeterministic, and the walkthrough must not pretend it paused."
    )
print(f"\nDurable run directory: {RUN_DIR}")

## 6. Verify the pinned run envelope

**Actors:** gate orchestrator, Git, and Codex runtime adapter.

The run stores the commit and the authoritative brief verbatim. It also records the requested model, reasoning effort, invocation role, attempt number, and installed Codex CLI version before each model call. These checks establish the provenance of the pinned inputs and the two model processes used during `start`; the next cell shows the exact rendered coding prompt.


In [ ]:
pinned_commit = str(START_SNAPSHOT["base_commit"])
pinned_brief = run_command(
    ["git", "show", f"{pinned_commit}:{BRIEF_PATH}"], show_output=False
).stdout
pinned_schema = run_command(
    ["git", "show", f"{pinned_commit}:{SCHEMA_PATH}"], show_output=False
).stdout

envelope = {
    "run_id": START_SNAPSHOT["run_id"],
    "state": START_SNAPSHOT["state"],
    "base_commit": pinned_commit,
    "created_at": START_SNAPSHOT["created_at"],
    "updated_at": START_SNAPSHOT["updated_at"],
    "original_brief": START_SNAPSHOT["original_brief"],
    "model_invocations": START_SNAPSHOT["model_invocations"],
    "brief_matches_pinned_commit": START_SNAPSHOT["original_brief"] == pinned_brief,
    "schema_matches_pre_start_commit": pinned_schema == schema_at_head,
}
print(json.dumps(envelope, indent=2))
attempt_one_prompt = str(START_SNAPSHOT["attempts"][0]["coding_prompt"])
assert pinned_commit == HEAD_BEFORE_START
assert pinned_brief in attempt_one_prompt
assert pinned_schema in attempt_one_prompt
assert [record["role"] for record in envelope["model_invocations"]] == [
    "CODING_AGENT",
    "EVIDENCE_REVIEWER",
]
assert all(record["model"] == CODEX_MODEL for record in envelope["model_invocations"])
assert all(
    record["reasoning_effort"] == CODEX_REASONING_EFFORT for record in envelope["model_invocations"]
)
assert envelope["brief_matches_pinned_commit"]
assert envelope["schema_matches_pre_start_commit"]

## 7. Inspect the first coding request

**Actor:** gate prompt renderer.

This is the exact project-controlled prompt persisted for attempt one. It is the materialized execution envelope described above: isolation instructions followed by the original brief and baseline schema. The product requirements occur only inside the verbatim brief. The coding agent receives no typed answer for the omitted existing-link policy.


In [ ]:
from implicit_decision_gate.codex_client import CODING_SCHEMA

attempt_one = START_SNAPSHOT["attempts"][0]
print("Persisted coding prompt:\n")
print(attempt_one["coding_prompt"])
print("\nCodex structured-output schema:\n")
print(json.dumps(CODING_SCHEMA, indent=2))

## 8. Inspect the first proposed migration

**Actors:** coding agent for the proposal, run store for the immutable artifact, and Git worktree manager for isolation.

The migration has three useful representations in this system: SQL as the proposed mechanism, a hash-identified immutable artifact, and normalized behavior produced by PostgreSQL. This cell shows the first two and independently recomputes the artifact digest.


In [ ]:
artifact_one_path = RUN_DIR / "attempt-1.sql"
migration_one = artifact_one_path.read_text(encoding="utf-8")
computed_digest_one = hashlib.sha256(migration_one.encode()).hexdigest()
worktree_one = Path(str(attempt_one["worktree_path"]))
worktree_one_head = run_command(
    ["git", "-C", str(worktree_one), "rev-parse", "HEAD"],
    show_output=False,
).stdout.strip()
worktree_one_status = run_command(
    ["git", "-C", str(worktree_one), "status", "--short"],
    show_output=False,
).stdout.rstrip()

print("Attempt-one SQL artifact:\n")
print(migration_one)
artifact_one = {
    "path": str(artifact_one_path),
    "stored_digest": attempt_one["migration_digest"],
    "computed_digest": computed_digest_one,
    "digest_matches": attempt_one["migration_digest"] == computed_digest_one,
    "file_mode": oct(stat.S_IMODE(artifact_one_path.stat().st_mode)),
    "worktree_path": str(worktree_one),
    "clean_start_verified_before_write": attempt_one["clean_start_verified"],
    "worktree_head": worktree_one_head,
    "worktree_matches_base_commit": worktree_one_head == pinned_commit,
    "worktree_status_after_write": worktree_one_status,
}
print(json.dumps(artifact_one, indent=2))
assert artifact_one["digest_matches"]
assert artifact_one["worktree_matches_base_commit"]

## 9. Inspect normalized runtime evidence

**Actor:** PostgreSQL verifier.

The verifier applied the baseline schema and migration in a disposable database, inspected the column and two representative rows, then rolled the transaction back. A modeled policy requires the exact PostgreSQL type, a nullable column, a non-null default, and one immediate insert whose expiration is within 10 seconds of migration time plus 30 days. The existing-row label then distinguishes the two accepted policies; anything else is `UNMODELED` and fails before evidence review.

The raw timestamps exist only during the probe; `run.json` retains the normalized facts below. This bounded check does not prove how an arbitrary future link relates to its own `created_at`. The `ProbeResult` is the migration's behavioral representation for this demo contract.


In [ ]:
probe_one = attempt_one["probe_result"]
print(json.dumps(probe_one, indent=2))

## 10. Inspect the narrow evidence review

**Actor:** evidence reviewer, running in a separate fresh Codex process.

The reviewer does not analyze every meaning in the brief. It receives the exact brief plus one observed rollout hypothesis and asks whether that behavior is explicitly supported. Its structured-output schema requires a `classification` from `SUPPORTED`, `CONTRADICTED`, `NOT_EVIDENCED`, or `UNCERTAIN`, plus an `evidence_quote` string, with no additional fields. The adapter stores an empty quote as `null`.

The persisted AI-derived result is only `{classification, evidence_quote}`; supported or contradicted quotes are additionally checked as literal substrings of the brief.


In [ ]:
print("Persisted reviewer prompt:\n")
print(START_SNAPSHOT["reviewer_prompt"])
print("\nValidated reviewer result:\n")
print(json.dumps(START_SNAPSHOT["reviewer_result"], indent=2))

# This is an inspection view over persisted fields, not another stored model.
brief_after_review_view = {
    "authoritative_brief": START_SNAPSHOT["original_brief"],
    "observed_hypothesis": START_SNAPSHOT["decision"]["observed"],
    "reviewer_result": START_SNAPSHOT["reviewer_result"],
}
print("\nWhat exists after review:\n")
print(json.dumps(brief_after_review_view, indent=2))

### Gate logic

**Actor:** deterministic decision logic inside the gate orchestrator.

The reviewer does not decide the rollout policy. Its classification determines whether the observed behavior is already justified, conflicts with the brief, or exposes missing intent. Only the missing-intent branch asks the owner.

```mermaid
flowchart TD
    probe[Normalized PostgreSQL probe result]
    modeled{Modeled rollout option}
    unmodeled[FAILED]
    review[Evidence reviewer classification]
    reviewResult{Evidence classification}
    completeFirst[COMPLETED]
    conflict[FAILED]
    pause[AWAITING_OWNER]
    answer[Owner selects typed option]
    ready[READY_TO_RESUME]
    retry[Fresh coding attempt and PostgreSQL probe]
    equal{Observed option equals selected option}
    completeSecond[COMPLETED]
    mismatch[FAILED]

    probe --> modeled
    modeled -->|UNMODELED| unmodeled
    modeled -->|PRESERVE_EXISTING or EXPIRE_EXISTING| review
    review --> reviewResult
    reviewResult -->|SUPPORTED| completeFirst
    reviewResult -->|CONTRADICTED| conflict
    reviewResult -->|NOT_EVIDENCED or UNCERTAIN| pause
    pause --> answer
    answer --> ready
    ready --> retry
    retry --> equal
    equal -->|Yes| completeSecond
    equal -->|No| mismatch
```


## 11. Inspect the durable pause and typed question

**Actor:** deterministic gate.

`NOT_EVIDENCED` or `UNCERTAIN` maps to `AWAITING_OWNER`. The fixed application vocabulary turns the observed policy into a narrow decision request with two verifiable choices. `decision_request` is derived for presentation; `run.json` persists the smaller `DecisionRecord`.


In [ ]:
pause_summary = run_idg("show", RUN_ID)
print("\nPersisted decision record:\n")
print(json.dumps(START_SNAPSHOT["decision"], indent=2))
assert pause_summary["state"] == "AWAITING_OWNER"
assert pause_summary["decision_request"] is not None

## 12. Complete the missing contract

**Actor:** human job owner.

Inspect the observed option and the two behaviors above, then review or edit the literal below. Either choice is valid: selecting the observed option explicitly confirms it, while selecting the other option makes a behavioral change easier to see in the SQL diff. The checked-in run records `EXPIRE_EXISTING`; the value is an explicit owner input and is not derived by a model or orchestration rule.


In [ ]:
OBSERVED_OPTION = str(START_SNAPSHOT["decision"]["observed"])

# Human decision: edit this one value after reading the decision request.
OWNER_OPTION = "EXPIRE_EXISTING"

valid_owner_options = {"PRESERVE_EXISTING", "EXPIRE_EXISTING"}
if OWNER_OPTION not in valid_owner_options:
    raise ValueError(f"OWNER_OPTION must be one of {sorted(valid_owner_options)}")
policy_changed = OWNER_OPTION != OBSERVED_OPTION
print(f"Observed: {OBSERVED_OPTION}")
print(f"Owner selected: {OWNER_OPTION}")
print(f"Behavioral policy changed: {policy_changed}")
if not policy_changed:
    print("The owner confirmed the observed policy; this is a valid contract completion.")

**Actor:** human job owner, with the run store persisting the answer.

`answer` accepts only one of the two typed options. It records the decision and advances the durable state to `READY_TO_RESUME`; it does not call a model or interpret free text. This prototype trusts the local caller and does not authenticate or attribute the owner identity; a production integration must supply that control.


In [ ]:
answer_summary = run_idg("answer", RUN_ID, "--option", OWNER_OPTION)
ANSWER_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
print("\nPersisted owner decision:\n")
print(json.dumps(ANSWER_SNAPSHOT["decision"], indent=2))
assert answer_summary["state"] == "READY_TO_RESUME"

## 13. Resume from durable state

**Actors:** gate orchestrator and a new coding-agent process.

`resume` can run later or in another process. It loads the recorded answer, creates a second clean worktree at the original commit, starts a fresh ephemeral Codex process, and probes the regenerated migration. The first model process is not resumed.


In [ ]:
resume_summary = run_idg("resume", RUN_ID)
FINAL_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
attempt_two = FINAL_SNAPSHOT["attempts"][1]
assert len(FINAL_SNAPSHOT["attempts"]) == 2
assert [record["role"] for record in FINAL_SNAPSHOT["model_invocations"]] == [
    "CODING_AGENT",
    "EVIDENCE_REVIEWER",
    "CODING_AGENT",
]
assert FINAL_SNAPSHOT["model_invocations"][-1]["attempt_number"] == 2
assert all(record["model"] == CODEX_MODEL for record in FINAL_SNAPSHOT["model_invocations"])
assert all(
    record["reasoning_effort"] == CODEX_REASONING_EFFORT
    for record in FINAL_SNAPSHOT["model_invocations"]
)

## 14. Inspect the fresh second request

**Actor:** gate prompt renderer supplying the new coding-agent process.

Attempt two receives the original brief and schema plus the authoritative owner choice, its required behavior, and PostgreSQL-specific acceptance criteria. It does not receive attempt one's SQL or the reviewer's rationale.


In [ ]:
prompt_two = str(attempt_two["coding_prompt"])
print("Persisted attempt-two coding prompt:\n")
print(prompt_two)

isolation_checks = {
    "different_worktree": attempt_two["worktree_path"] != attempt_one["worktree_path"],
    "second_clean_start_verified": attempt_two["clean_start_verified"],
    "attempt_one_sql_absent": migration_one.strip() not in prompt_two,
    "reviewer_prompt_absent": START_SNAPSHOT["reviewer_prompt"] not in prompt_two,
    "owner_decision_present": f"Authoritative owner decision: {OWNER_OPTION}" in prompt_two,
}
print("\nContext isolation checks:\n")
print(json.dumps(isolation_checks, indent=2))
assert all(isolation_checks.values())

## 15. Compare the two proposed mechanisms

**Actors:** second coding agent for the new SQL and notebook operator acting as auditor.

The unified diff makes the agent's implementation change inspectable. It is not the correctness proof: that comes from the second PostgreSQL probe in the next stage. The metadata also demonstrates separate worktrees and immutable digests.


In [ ]:
artifact_two_path = RUN_DIR / "attempt-2.sql"
migration_two = artifact_two_path.read_text(encoding="utf-8")
computed_digest_two = hashlib.sha256(migration_two.encode()).hexdigest()
migration_diff = "".join(
    difflib.unified_diff(
        migration_one.splitlines(keepends=True),
        migration_two.splitlines(keepends=True),
        fromfile="attempt-1.sql",
        tofile="attempt-2.sql",
    )
)

print("Attempt-two SQL artifact:\n")
print(migration_two)
print("Unified diff:\n")
print(migration_diff or "No textual difference between the two migrations.")
comparison = {
    "attempt_1": {
        "digest": attempt_one["migration_digest"],
        "worktree": attempt_one["worktree_path"],
    },
    "attempt_2": {
        "digest": attempt_two["migration_digest"],
        "computed_digest": computed_digest_two,
        "digest_matches": attempt_two["migration_digest"] == computed_digest_two,
        "worktree": attempt_two["worktree_path"],
    },
}
print(json.dumps(comparison, indent=2))
assert comparison["attempt_2"]["digest_matches"]

## 16. Verify the completed contract

**Actors:** PostgreSQL verifier first, then deterministic gate.

PostgreSQL again reduces runtime behavior to the bounded `ProbeResult`. Final acceptance is the typed equality shown below: the observed rollout must equal the owner's selected rollout. No model judges whether attempt two succeeded.


In [ ]:
probe_two = attempt_two["probe_result"]
selected_option = str(FINAL_SNAPSHOT["decision"]["selected"])
observed_option_two = str(probe_two["rollout_option"])
selected_matches_observed = selected_option == observed_option_two
final_verification = {
    "selected_by_owner": selected_option,
    "observed_by_postgresql": observed_option_two,
    "selected_equals_observed": selected_matches_observed,
    "final_state": FINAL_SNAPSHOT["state"],
    "error": FINAL_SNAPSHOT["error"],
    "probe_result": probe_two,
}
print(json.dumps(final_verification, indent=2))
assert selected_matches_observed
assert resume_summary["state"] == "COMPLETED"

## 17. Inspect the durable record

**Actor:** auditor reading the run store.

The complete `run.json` below is the atomically replaced current-state snapshot. It is not an append-only event log or a hidden graph. Its ordered model invocations, attempts, decision timestamps, prompts, probe results, and adjacent immutable SQL files are the inspectable state of this run.


In [ ]:
run_json_path = RUN_DIR / "run.json"
print("Complete run.json:\n")
print(run_json_path.read_text(encoding="utf-8"))

timeline = [
    {"event": "run_created", "at": FINAL_SNAPSHOT["created_at"]},
    {
        "event": "attempt_1_completed",
        "at": FINAL_SNAPSHOT["attempts"][0]["completed_at"],
    },
    {"event": "owner_answered", "at": FINAL_SNAPSHOT["decision"]["answered_at"]},
    {
        "event": "attempt_2_completed",
        "at": FINAL_SNAPSHOT["attempts"][1]["completed_at"],
    },
    {"event": "snapshot_updated", "at": FINAL_SNAPSHOT["updated_at"]},
]
artifacts = [
    {"name": path.name, "bytes": path.stat().st_size}
    for path in sorted(RUN_DIR.glob("attempt-*.sql"))
]
print("Derived timeline:\n")
print(json.dumps(timeline, indent=2))
print("\nImmutable SQL artifacts:\n")
print(json.dumps(artifacts, indent=2))

## 18. Current boundary

**Actor:** system designer.

What exists in this vertical slice:

- A commit-pinned authoritative brief and schema.
- Application-pinned `gpt-5.6-terra` model calls at `xhigh` with per-invocation provenance.
- A coding agent constrained to structured SQL output.
- A real PostgreSQL observation normalized into one typed policy vocabulary.
- A narrow AI evidence review of one observed hypothesis.
- A durable typed human amendment and clean-context retry.
- A deterministic final comparison between selected and observed behavior.

What does not exist yet:

- A general semantic compiler that turns arbitrary prose into a trusted typed contract.
- Authenticated and attributable agent or human identity.
- Persisted raw database observations or raw model transcripts.
- An append-only event ledger or graph database.

A future semantic verifier could let AI propose typed claims, unknown fields, and source spans for human approval, then compare the approved policy with normalized runtime evidence. The AI-derived interpretation must not silently replace the human-owned brief. The current reviewer is also deliberately weaker than that vision: literal quote validation proves source occurrence, not semantic relevance.


## 19. Optional infrastructure cleanup

**Actor:** notebook operator.

Stopping Compose removes the local verifier container after the walkthrough. It does not remove `.idg/runs/<run_id>`, the SQL artifacts, or the detached worktrees, so the evidence remains available for manual inspection. Set the flag to `True` when you are finished.


In [ ]:
STOP_DOCKER = False

if EXTERNAL_POSTGRES:
    print("No Compose-managed PostgreSQL instance was started.")
elif STOP_DOCKER:
    run_command(["docker", "compose", "down"])
else:
    print("PostgreSQL is still running. Set STOP_DOCKER = True and rerun this cell to stop it.")

## 20. How the pattern scales

**Actor:** system designer.

The reusable idea is not this particular migration rule and it is not a universal LLM judge. It is a contract-completion interface:

```text
authoritative contract
→ agent action
→ trusted runtime observation
→ normalized typed outcome
→ deterministic comparison
→ owner decision for any unknown
```

For PostgreSQL, additional evidence adapters could verify backfills, nullability changes, uniqueness and foreign-key constraints, deletion behavior, indexes, transactional safety, rollback behavior, and compatibility with representative existing rows. Concurrency, locking, and performance claims would require dedicated workload probes rather than being inferred from SQL text.

The same lifecycle can extend to other surfaces:

| Surface | Trusted observation | Example typed facts |
| --- | --- | --- |
| REST or GraphQL APIs | Sandboxed contract requests and schema comparison | Response schema, status behavior, authorization, compatibility, and side effects |
| Events and queues | Test broker, schema registry, and recorded consumer behavior | Schema version, routing, ordering, delivery, and idempotency |
| Infrastructure | Plan output plus cloud control-plane observations | Resource changes, network exposure, encryption, and retention |
| Builds and deployments | Signed artifacts, test results, and rollout telemetry | Artifact digest, required checks, health, and rollback outcome |
| Permissions | Policy-engine decisions and attributable audit evidence | Principal, action, resource, conditions, and granted capability |

The durable state machine, evidence provenance, typed owner decision, clean retry, and deterministic comparison can be reused across these surfaces. Each surface still needs its own trusted observer, normalization adapter, and bounded vocabulary. AI can help extract proposed claims or classify source evidence, but it should not be the sole authority for either the contract or the observed truth.

In a 1Password integration, Verified Loops would continue to provide authenticated human and agent identity, controlled tool access, attributable evidence, and permission enforcement. This stage would only complete missing intent and return the amended contract and verified result to that wider architecture.
